## KRP - PC application
- PC Application for the USB project on the STM32H747I-DISCO board
- Github: https://github.com/pavlio12/KRP_Project


In [ ]:
from connection_manager import ConnectionManager
from sender import sender_loop
from gui import create_gui

from queue import Queue, Empty
import threading
from collections import deque
import threading, time
from IPython.display import Javascript, display
from IPython import get_ipython
import tornado.ioloop

message_queue = Queue()  # Thread-safe queue for messages to send

### Configuration

In [ ]:
# === Configuration ===
PORT = "COM18"      # Windows example
BAUDRATE = 115200   # default CDC speed

In [ ]:
# ----- LOG STATE (one buffer, one lock) -----
_LOG_MAX = 500  # keep it bounded so the notebook doesn’t bloat
_log_lines = deque(maxlen=_LOG_MAX)
_log_lock = threading.Lock()
_LOG_Q = Queue()

def _enqueue(line: str):
    # Safe to call from ANY thread (receiver, sender, whatever)
    _LOG_Q.put(line)

# Drainer: runs on the kernel I/O loop thread (the one Jupyter listens to)
def _drain_queue():
    lines = []
    try:
        while True:
            lines.append(_LOG_Q.get_nowait())
    except Empty:
        pass
    if lines:
        # ONE print per drain to reduce flicker / overhead
        print("\n".join(lines), flush=True)

def log_periodically():
    """Set up periodic draining of the log queue to the notebook output."""
    ip = get_ipython()
    if hasattr(ip, "kernel") and hasattr(ip.kernel, "io_loop"):
        _pc = tornado.ioloop.PeriodicCallback(_drain_queue, 50)  # every 50 ms
        _pc.start()
        _pc_started = True
    else:
        # Fallback if somehow not in ipykernel (rare): poll via a timer thread
        import threading
        def _fallback_poller():
            _drain_queue()
            t = threading.Timer(0.05, _fallback_poller)
            t.daemon = True
            t.start()
        _fallback_poller()

### Main

In [ ]:
def main():
    # === GUI Callbacks ===
    def on_status_change(msg):
        """Update the status label in the GUI."""
        status_label.value = msg

    # def on_receive(msg):
    #     """Display received messages in the output area."""
    #     output_area.append_stdout(f"[Rx] {msg}\n")

    # def on_tx(msg):
    #     """Display transmitted messages in the output area."""
    #     output_area.append_stdout(f"[Tx] {msg}\n")

    def on_receive(msg):
        ts = time.strftime("%H:%M:%S")
        _enqueue(f"[{ts}] RX {msg}")

    def on_tx(msg):
        ts = time.strftime("%H:%M:%S")
        _enqueue(f"[{ts}] TX {msg}")

    # === Connection Management ===
    def start_connection():
        global conn_manager
        conn_manager = ConnectionManager(PORT, BAUDRATE, on_status_change, on_receive)
        conn_manager.start()
        threading.Thread(target=sender_loop, args=(conn_manager, message_queue, on_tx), daemon=True).start()

    def stop_connection():
        global conn_manager
        if conn_manager:
            conn_manager.stop()
            conn_manager = None  # Clear the reference to avoid reuse
            status_label.value = "Disconnected"

    # === GUI Initialization ===
    gui, status_label = create_gui(start_connection, stop_connection, message_queue.put)
    display(gui)


# Execute
main()
log_periodically()

